# BCCD: подсчет клеток и OOD-детекция

Цель: решить задачу подсчета `RBC`, `WBC`, `Platelets` на изображениях BCCD двумя способами: прямой регрессией количества и подсчетом детекций YOLO. Отдельно проверяем OOD для размытых, зашумленных и внешних не-кровяных изображений.

Основной код вынесен в `main.py`, чтобы эксперимент можно было запускать из ноутбука и из терминала одинаково.

## Установка

Если зависимости еще не установлены, выполните в терминале из папки `practice/neural_networks/bccd`:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -U pip
pip install -e .
```

На CUDA пайплайн сам выберет GPU при `--device auto`.

In [ ]:
import os
from pathlib import Path

import torch

ROOT = Path.cwd()
if not (ROOT / "main.py").exists():
    candidate = (ROOT / "practice/neural_networks/bccd").resolve()
    if candidate.exists():
        os.chdir(candidate)
        ROOT = candidate
print(ROOT)
print("cuda:", torch.cuda.is_available())
print("mps:", torch.backends.mps.is_available())

## Анализ и визуализация BCCD

Скачиваем датасет, парсим Pascal VOC XML, получаем табличку с количеством каждого класса и сохраняем графики в `bccd_work/figures/`.

In [ ]:
from main import analyze_and_visualize, build_dataframe, ensure_bccd

data_dir = Path("data")
work_dir = Path("bccd_work")
work_dir.mkdir(exist_ok=True)

bccd_root = ensure_bccd(data_dir)
df = build_dataframe(bccd_root, work_dir / "bccd_dataset.csv")
analyze_and_visualize(df, work_dir / "figures")
df.head()

In [ ]:
df[["RBC", "WBC", "Platelets", "total_cells", "width", "height"]].describe().round(2)

## Способ A: прямое предсказание количества

Таргет для каждого изображения: вектор `[RBC, WBC, Platelets]`. Модель: `ResNet50` как извлекатель признаков и маленькая регрессионная голова. Для быстрого запуска backbone по умолчанию использует ImageNet-веса и заморожен; для более сильной модели можно добавить `--unfreeze-backbone`.

In [ ]:
# Быстрый прогон без YOLO: регрессия + OOD.
# В терминале это же запускается так: python main.py --skip-yolo
%run main.py --skip-yolo --epochs 3 --batch-size 32 --img-size 224

## Способ B: подсчет по детекции YOLO26

Для YOLO XML-разметка конвертируется в формат `class x_center y_center width height`. После обучения `yolo26n` считаем число предсказанных bbox каждого класса на validation split и сравниваем с истинными счетчиками теми же метриками: MAE, RMSE, R2 и exact match. Старый baseline можно включить аргументом `--yolo-model yolov8n.pt`.

In [ ]:
# Полный запуск с YOLO26. На CPU будет заметно медленнее, поэтому лучше CUDA.
# %run main.py --epochs 3 --yolo-epochs 10 --batch-size 32

## OOD-детекция

OOD-score считается по признакам ResNet50: на train-fit вычисляются среднее и стандартное отклонение признаков, затем для новых изображений используется средний квадрат z-score. Порог берется как 95-й перцентиль train-score.

Проверяются группы:
- clean validation images;
- Gaussian blur;
- Gaussian noise;
- external non-blood images из `data/external_ood/`, если они есть.

Если внешней папки нет, код создает синтетические не-кровяные текстуры, чтобы эксперимент оставался полностью исполняемым. Для более реалистичного пункта задания положите в `data/external_ood/` изображения бактерий, тканей или других микроскопических препаратов.

In [ ]:
import json

metrics_path = Path("bccd_work/metrics.json")
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics
else:
    print("Запустите одну из ячеек обучения выше, чтобы получить metrics.json")

## Что сравнивать в отчете

1. Распределения классов и примеры bbox из `figures/class_distribution.png` и `figures/annotated_samples.png`.
2. Метрики `regression_resnet50` против `detection_yolo26n_counting`.
3. Для регрессии важно округлять и обрезать предсказания до неотрицательных чисел, потому что ответом является количество клеток.
4. Для YOLO итоговый count получается не из отдельной модели, а из числа детекций каждого класса после confidence/IoU filtering.
5. В OOD смотрим `ood_rate` и `roc_auc_vs_clean`: blur/noise/external должны получать score выше clean.